In [1]:
# Manipulação e análise de dados
from pathlib import Path
import gc
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Machine Learning: métricas de avaliação
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Deep Learning: LSTM e GRU
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# Reprodutibilidade
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Definir o dispositivo de treinamento
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("PyTorch:", torch.__version__)
print("Dispositivo:", device)

NumPy: 2.5.4
Pandas: 3.1.0
PyTorch: 2.15.0+cu132
Dispositivo: cpu


In [2]:
# Caminhos possíveis do dataset
candidatos = [
    Path.home() / "Downloads" / "archive" / "SAML-D.csv",
    Path.home() / "Downloads" / "SAML-D.csv"
]

caminho = next(
    (arquivo for arquivo in candidatos if arquivo.is_file()),
    None
)

if caminho is None:
    raise FileNotFoundError(
        "SAML-D.csv não encontrado. Confira se o arquivo está em "
        "Downloads/archive/SAML-D.csv ou Downloads/SAML-D.csv."
    )

# Tipos de dados para reduzir o uso de memória
tipos = {
    "Time": "string",
    "Date": "string",
    "Sender_account": "int64",
    "Receiver_account": "int64",
    "Amount": "float64",
    "Payment_currency": "category",
    "Received_currency": "category",
    "Sender_bank_location": "category",
    "Receiver_bank_location": "category",
    "Payment_type": "category",
    "Is_laundering": "int8",
    "Laundering_type": "category"
}

df = pd.read_csv(
    caminho,
    dtype=tipos,
    low_memory=False
)

# Criar o campo temporal
df["timestamp"] = pd.to_datetime(
    df["Date"].astype(str) + " " + df["Time"].astype(str),
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

print("Arquivo:", caminho)
print("Dimensões:", df.shape)
print("Datas distintas:", df["Date"].nunique())
print("Datas inválidas:", df["timestamp"].isna().sum())

print("\nDistribuição das classes:")
display(df["Is_laundering"].value_counts().to_frame("Quantidade"))

print("\nPrimeiras transações:")
display(df.head())

print(
    "\nMemória do DataFrame:",
    round(df.memory_usage(deep=True).sum() / 1024**3, 2),
    "GB"
)

Arquivo: C:\Users\Aluno\Downloads\archive\SAML-D.csv
Dimensões: (9504852, 13)
Datas distintas: 321
Datas inválidas: 0

Distribuição das classes:


,Quantidade
Is_laundering,
0,9494979
1,9873



Primeiras transações:


,Time,Date,Sender_account,Receiver_account,Amount,Payment_currency,Received_currency,Sender_bank_location,Receiver_bank_location,Payment_type,Is_laundering,Laundering_type,timestamp
0,10:35:19,2022-10-07,8724731955,2769355426,1459.15,UK pounds,UK pounds,UK,UK,Cash Deposit,0,Normal_Cash_Deposits,2022-10-07 10:35:19
1,10:35:20,2022-10-07,1491989064,8401255335,6019.64,UK pounds,Dirham,UK,UAE,Cross-border,0,Normal_Fan_Out,2022-10-07 10:35:20
2,10:35:20,2022-10-07,287305149,4404767002,14328.44,UK pounds,UK pounds,UK,UK,Cheque,0,Normal_Small_Fan_Out,2022-10-07 10:35:20
3,10:35:21,2022-10-07,5376652437,9600420220,11895.00,UK pounds,UK pounds,UK,UK,ACH,0,Normal_Fan_In,2022-10-07 10:35:21
4,10:35:21,2022-10-07,9614186178,3803336972,115.25,UK pounds,UK pounds,UK,UK,Cash Deposit,0,Normal_Cash_Deposits,2022-10-07 10:35:21



Memória do DataFrame: 1.37 GB


In [3]:
# Quantidade de transações por conta
def resumo_contas(coluna):

    quantidade = df.groupby(
        coluna,
        observed=True,
        sort=False
    ).size()

    return pd.Series({
        "Contas distintas": quantidade.size,
        "Mediana de transações": quantidade.median(),
        "Média de transações": quantidade.mean(),
        "Percentil 90": quantidade.quantile(0.90),
        "Percentil 99": quantidade.quantile(0.99),
        "Máximo de transações": quantidade.max(),
        "Contas com uma transação": (quantidade == 1).sum()
    })


# Comparar remetentes e destinatários
resumo = pd.DataFrame({
    "Como remetente": resumo_contas("Sender_account"),
    "Como destinatário": resumo_contas("Receiver_account")
})

display(resumo.round(2))


# Identificar contas que aparecem nos dois papéis
remetentes = pd.Index(df["Sender_account"].unique())
destinatarios = pd.Index(df["Receiver_account"].unique())

contas_comuns = remetentes.intersection(destinatarios)
contas_unicas = remetentes.union(destinatarios)

print("=== DISTRIBUIÇÃO GERAL DAS CONTAS ===")
print("Contas como remetente:", len(remetentes))
print("Contas como destinatário:", len(destinatarios))
print("Contas presentes nos dois papéis:", len(contas_comuns))
print("Total de contas distintas:", len(contas_unicas))

,Como remetente,Como destinatário
Contas distintas,292715.00,652266.00
Mediana de transações,12.00,13.00
Média de transações,32.47,14.57
Percentil 90,14.00,18.00
Percentil 99,506.00,191.00
Máximo de transações,754.00,751.00
Contas com uma transação,27816.00,92413.00


=== DISTRIBUIÇÃO GERAL DAS CONTAS ===
Contas como remetente: 292715
Contas como destinatário: 652266
Contas presentes nos dois papéis: 89521
Total de contas distintas: 855460


In [4]:
# Quantidade original de transações
n = len(df)

# Codificar os tipos de pagamento
payment_codes = df["Payment_type"].cat.codes.to_numpy(
    dtype=np.int8
)

# Identificar as classes das transações
y = df["Is_laundering"].to_numpy(dtype=np.int8)

# Criar os eventos de envio e recebimento
eventos_conta = pd.DataFrame({

    # Identificador da transação original
    "transaction_id": np.tile(
        np.arange(n, dtype=np.int32), 2
    ),

    # Conta que estamos analisando
    "account_id": np.concatenate([
        df["Sender_account"].to_numpy(),
        df["Receiver_account"].to_numpy()
    ]),

    # Outra conta envolvida na transação
    "counterparty_id": np.concatenate([
        df["Receiver_account"].to_numpy(),
        df["Sender_account"].to_numpy()
    ]),

    # Data e hora
    "timestamp": np.concatenate([
        df["timestamp"].to_numpy(),
        df["timestamp"].to_numpy()
    ]),

    # Valor da transação
    "amount": np.concatenate([
        df["Amount"].to_numpy(dtype=np.float32),
        df["Amount"].to_numpy(dtype=np.float32)
    ]),

    # Tipo de pagamento codificado
    "payment_type": np.concatenate([
        payment_codes,
        payment_codes
    ]),

    # 1 = ENVIO | 0 = RECEBIMENTO
    "direction": np.concatenate([
        np.ones(n, dtype=np.int8),
        np.zeros(n, dtype=np.int8)
    ]),

    # Rótulo original da transação
    "y": np.concatenate([y, y])
})

print("Eventos criados:", len(eventos_conta))
print("Contas distintas:", eventos_conta["account_id"].nunique())

print("\nDistribuição dos eventos:")
print(eventos_conta["direction"].value_counts())

print("\nDistribuição dos rótulos:")
print(eventos_conta["y"].value_counts())

Eventos criados: 19009704
Contas distintas: 855460

Distribuição dos eventos:
direction
1    9504852
0    9504852
Name: count, dtype: int64

Distribuição dos rótulos:
y
0    18989958
1       19746
Name: count, dtype: int64


In [5]:
print("Ordenando eventos por conta e horário...")

eventos_conta = eventos_conta.sort_values(
    ["account_id", "timestamp", "transaction_id"],
    kind="stable",
    ignore_index=True
)

print("Ordenação concluída!")

display(
    eventos_conta.head(10)
)

Ordenando eventos por conta e horário...
Ordenação concluída!


,transaction_id,account_id,counterparty_id,timestamp,amount,payment_type,direction,y
0,8562799,9018,6080420349,2023-07-21 21:32:35,3319.060059,0,0,0
1,8572082,9018,2388293593,2023-07-22 09:51:28,3319.060059,5,1,0
2,1420882,13266,2595228252,2022-11-24 16:20:16,1178.699951,3,0,0
3,1753672,13266,2595228252,2022-12-06 04:28:33,2218.120117,4,0,0
4,2850709,13266,2595228252,2023-01-12 19:44:39,594.969971,4,0,0
5,3186386,13266,2595228252,2023-01-24 09:13:08,1043.719971,3,0,0
6,3290478,13266,2595228252,2023-01-27 18:08:10,674.909973,3,0,0
7,3346265,13266,2595228252,2023-01-29 16:57:38,650.840027,0,0,0
8,3768458,13266,2595228252,2023-02-11 21:30:07,1597.339966,0,0,0
9,5023851,13266,2595228252,2023-03-25 01:51:52,1137.540039,4,0,0


In [6]:
# Encontrar uma conta com bastante movimentação
contagem = eventos_conta["account_id"].value_counts()

conta_exemplo = contagem.index[0]

# Recuperar todos os eventos dessa conta
historico_conta = eventos_conta.loc[
    eventos_conta["account_id"] == conta_exemplo
].copy()

# Traduzir os códigos para facilitar a leitura
historico_conta["tipo_evento"] = (
    historico_conta["direction"]
    .map({1: "ENVIO", 0: "RECEBIMENTO"})
)

categorias_pagamento = dict(
    enumerate(df["Payment_type"].cat.categories)
)

historico_conta["payment_type_nome"] = (
    historico_conta["payment_type"]
    .map(categorias_pagamento)
)

print("Conta analisada:", conta_exemplo)
print("Quantidade de eventos:", len(historico_conta))

display(
    historico_conta[
        [
            "timestamp",
            "tipo_evento",
            "counterparty_id",
            "amount",
            "payment_type_nome",
            "y"
        ]
    ].head(20)
)

Conta analisada: 2938210715
Quantidade de eventos: 1498


,timestamp,tipo_evento,counterparty_id,amount,payment_type_nome,y
5572994,2022-10-07 21:18:10,RECEBIMENTO,4374280932,6209.709961,ACH,0
5572995,2022-10-07 23:31:16,ENVIO,3933096796,4983.959961,Cheque,0
5572996,2022-10-08 06:49:04,ENVIO,4567520117,13138.509766,Cheque,0
5572997,2022-10-08 21:41:39,RECEBIMENTO,7076837529,15733.599609,Cheque,0
5572998,2022-10-08 23:23:06,RECEBIMENTO,7076837529,15198.209961,ACH,0
5572999,2022-10-08 23:45:11,RECEBIMENTO,7076837529,21968.560547,Debit card,0
5573000,2022-10-09 07:09:49,ENVIO,3528690905,8480.629883,ACH,0
5573001,2022-10-09 11:20:23,RECEBIMENTO,3933096796,2482.090088,ACH,0
5573002,2022-10-09 17:19:56,ENVIO,4567520117,9205.120117,Debit card,0
5573003,2022-10-09 21:35:41,ENVIO,8074478839,3080.830078,Debit card,0


In [7]:
# Quantidade de eventos por conta
eventos_por_conta = (
    eventos_conta.groupby("account_id", sort=False)
    .size()
)

# Estatísticas da distribuição
print("=== DISTRIBUIÇÃO DE EVENTOS POR CONTA ===")

display(
    eventos_por_conta.describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    ).to_frame("Quantidade de eventos").round(2)
)

# Contas que possuem histórico suficiente
for minimo in [5, 10, 20, 40, 60, 100]:
    quantidade = (eventos_por_conta >= minimo).sum()

    print(
        f"Contas com pelo menos {minimo} eventos: "
        f"{quantidade:,} "
        f"({quantidade / len(eventos_por_conta) * 100:.2f}%)"
    )

# Estimativa de janelas não sobrepostas de 20 eventos
total_janelas = (eventos_por_conta // 20).sum()

print("\n=== ESTIMATIVA DE JANELAS ===")
print("Janelas de 20 eventos:", f"{total_janelas:,}")
print(
    "Contas com pelo menos uma janela:",
    f"{(eventos_por_conta >= 20).sum():,}"
)

=== DISTRIBUIÇÃO DE EVENTOS POR CONTA ===


,Quantidade de eventos
count,855460.00
mean,22.22
std,77.19
min,1.00
25%,9.00
50%,12.00
75%,13.00
90%,17.00
95%,21.00
99%,522.00


Contas com pelo menos 5 eventos: 738,109 (86.28%)
Contas com pelo menos 10 eventos: 617,878 (72.23%)
Contas com pelo menos 20 eventos: 54,663 (6.39%)
Contas com pelo menos 40 eventos: 25,937 (3.03%)
Contas com pelo menos 60 eventos: 23,566 (2.75%)
Contas com pelo menos 100 eventos: 23,474 (2.74%)

=== ESTIMATIVA DE JANELAS ===
Janelas de 20 eventos: 515,787
Contas com pelo menos uma janela: 54,663


In [8]:
import pandas as pd

tamanhos = [5, 10, 15, 20, 30]

resultados = []

total_contas = len(eventos_por_conta)
total_eventos = eventos_por_conta.sum()

for tamanho in tamanhos:

    # Quantidade de janelas completas por conta
    janelas = (eventos_por_conta // tamanho).sum()

    # Quantas contas conseguem formar pelo menos uma janela
    contas = (eventos_por_conta >= tamanho).sum()

    # Quantos eventos serão aproveitados
    eventos_usados = janelas * tamanho

    resultados.append({
        "Tamanho da janela": tamanho,
        "Contas com janela": contas,
        "Cobertura das contas (%)": contas / total_contas * 100,
        "Janelas completas": janelas,
        "Eventos aproveitados (%)": eventos_usados / total_eventos * 100
    })

comparacao_janelas = pd.DataFrame(resultados)

display(comparacao_janelas.round(2))

,Tamanho da janela,Contas com janela,Cobertura das contas (%),Janelas completas,Eventos aproveitados (%)
0,5,738109,86.28,3432189,90.27
1,10,617878,72.23,1607407,84.56
2,15,137210,16.04,764705,60.34
3,20,54663,6.39,515787,54.27
4,30,27113,3.17,320722,50.61


In [10]:
import gc
import pandas as pd

# Tamanho das janelas
TAMANHO_JANELA = 10

# Garantir que o timestamp seja datetime
eventos_conta["timestamp"] = pd.to_datetime(
    eventos_conta["timestamp"]
)

# Definir os limites como datas, não textos
inicio_validacao = pd.Timestamp("2023-05-19")
inicio_teste = pd.Timestamp("2023-07-06")

# Separar os períodos cronologicamente
periodos = {
    "Treino": eventos_conta["timestamp"] < inicio_validacao,

    "Validacao": (
        (eventos_conta["timestamp"] >= inicio_validacao)
        & (eventos_conta["timestamp"] < inicio_teste)
    ),

    "Teste": eventos_conta["timestamp"] >= inicio_teste
}

janelas_por_periodo = {}
resultados = []

# Construir as janelas
for nome, mascara in periodos.items():

    print(f"\nConstruindo janelas: {nome}...", flush=True)

    parte = eventos_conta.loc[
        mascara,
        ["account_id", "timestamp", "transaction_id", "y"]
    ].copy()

    # Numerar eventos dentro de cada conta
    parte["_posicao"] = (
        parte.groupby("account_id", sort=False)
        .cumcount()
    )

    # Criar janelas não sobrepostas de 10 eventos
    parte["_janela"] = (
        parte["_posicao"] // TAMANHO_JANELA
    ).astype("int32")

    # Agrupar os eventos
    janelas = (
        parte.groupby(
            ["account_id", "_janela"],
            sort=False
        )
        .agg(
            quantidade_eventos=("y", "size"),
            y=("y", "max"),
            suspeitas_na_janela=("y", "sum"),
            inicio=("timestamp", "first"),
            fim=("timestamp", "last"),
            primeira_transacao=("transaction_id", "first"),
            ultima_transacao=("transaction_id", "last")
        )
        .reset_index()
    )

    # Manter apenas janelas completas
    janelas = janelas[
        janelas["quantidade_eventos"] == TAMANHO_JANELA
    ].copy()

    janelas_por_periodo[nome] = janelas

    # Resumir os resultados
    total = len(janelas)
    positivas = int(janelas["y"].sum())
    negativas = total - positivas

    resultados.append({
        "Periodo": nome,
        "Contas": janelas["account_id"].nunique(),
        "Janelas": total,
        "Normais (0)": negativas,
        "Suspeitas (1)": positivas,
        "Suspeitas (%)": (
            positivas / total * 100 if total else 0
        )
    })

    print(f"Eventos analisados: {len(parte):,}")
    print(f"Janelas completas: {total:,}")
    print(f"Normais: {negativas:,}")
    print(f"Suspeitas: {positivas:,}")

    del parte
    gc.collect()

# Apresentar os resultados
resumo_janelas = pd.DataFrame(resultados)

print("\n=== RESUMO DAS JANELAS ===")
display(resumo_janelas.round(3))


Construindo janelas: Treino...
Eventos analisados: 13,322,446
Janelas completas: 1,078,404
Normais: 1,072,824
Suspeitas: 5,580

Construindo janelas: Validacao...
Eventos analisados: 2,861,610
Janelas completas: 194,016
Normais: 193,333
Suspeitas: 683

Construindo janelas: Teste...
Eventos analisados: 2,825,648
Janelas completas: 190,825
Normais: 190,039
Suspeitas: 786

=== RESUMO DAS JANELAS ===


,Periodo,Contas,Janelas,Normais (0),Suspeitas (1),Suspeitas (%)
0,Treino,414677,1078404,1072824,5580,0.517
1,Validacao,75628,194016,193333,683,0.352
2,Teste,74412,190825,190039,786,0.412


In [11]:
# Selecionar uma janela normal e uma suspeita do teste

janelas_teste = janelas_por_periodo["Teste"]

janela_normal = (
    janelas_teste[janelas_teste["y"] == 0]
    .sample(1, random_state=42)
    .iloc[0]
)

janela_suspeita = (
    janelas_teste[janelas_teste["y"] == 1]
    .sample(1, random_state=42)
    .iloc[0]
)

inicio_teste = pd.Timestamp("2023-07-06")

def visualizar_janela(janela, nome):

    conta = janela["account_id"]
    numero = janela["_janela"]

    # Recuperar os eventos da conta no período de teste
    eventos = eventos_conta.loc[
        (eventos_conta["account_id"] == conta)
        & (eventos_conta["timestamp"] >= inicio_teste)
    ]

    # Recuperar exatamente os 10 eventos da janela
    eventos = eventos.iloc[
        numero * TAMANHO_JANELA:
        (numero + 1) * TAMANHO_JANELA
    ].copy()

    eventos["tipo_evento"] = eventos["direction"].map({
        1: "ENVIO",
        0: "RECEBIMENTO"
    })

    eventos["payment_type_nome"] = (
        eventos["payment_type"]
        .map(dict(enumerate(df["Payment_type"].cat.categories)))
    )

    print(f"\n=== {nome} ===")
    print("Conta:", conta)
    print("Janela:", numero)
    print("Rótulo y:", janela["y"])
    print("Transações suspeitas:", int(eventos["y"].sum()))

    display(eventos[[
        "timestamp",
        "tipo_evento",
        "counterparty_id",
        "amount",
        "payment_type_nome",
        "y"
    ]])

visualizar_janela(janela_normal, "JANELA NORMAL")
visualizar_janela(janela_suspeita, "JANELA SUSPEITA")


=== JANELA NORMAL ===
Conta: 3283834928
Janela: 27
Rótulo y: 0
Transações suspeitas: 0


,timestamp,tipo_evento,counterparty_id,amount,payment_type_nome,y
6213741,2023-08-05 15:56:55,ENVIO,1885010244,9856.580078,ACH,0
6213742,2023-08-05 16:12:48,ENVIO,4118823855,5108.209961,Cross-border,0
6213743,2023-08-05 16:30:06,ENVIO,5555191008,10570.870117,ACH,0
6213744,2023-08-05 16:34:58,ENVIO,1885010244,9778.990234,Credit card,0
6213745,2023-08-05 16:35:48,ENVIO,5020459006,6408.370117,Credit card,0
6213746,2023-08-05 16:47:01,ENVIO,8950927875,4594.259766,Debit card,0
6213747,2023-08-05 16:59:33,ENVIO,2087869049,8921.629883,ACH,0
6213748,2023-08-05 16:59:46,ENVIO,9920262843,9539.809570,Cheque,0
6213749,2023-08-05 17:07:28,ENVIO,1459080469,7094.620117,ACH,0
6213750,2023-08-05 17:17:44,ENVIO,5020459006,6371.500000,Debit card,0



=== JANELA SUSPEITA ===
Conta: 9556816200
Janela: 2
Rótulo y: 1
Transações suspeitas: 1


,timestamp,tipo_evento,counterparty_id,amount,payment_type_nome,y
18166010,2023-08-08 21:37:20,ENVIO,2484095742,2303.949951,Credit card,0
18166011,2023-08-09 12:54:15,ENVIO,4609381415,155.660004,Debit card,0
18166012,2023-08-09 19:58:47,ENVIO,4609381415,128.899994,Debit card,0
18166013,2023-08-10 03:31:22,ENVIO,1906466788,895710.187500,ACH,0
18166014,2023-08-13 19:07:26,ENVIO,6673377477,85.070000,Credit card,0
18166015,2023-08-14 17:15:33,ENVIO,854160503,423.750000,Cheque,0
18166016,2023-08-15 08:51:45,ENVIO,583377608,687.969971,ACH,0
18166017,2023-08-16 17:56:39,RECEBIMENTO,6228181861,3151.228271,Cross-border,1
18166018,2023-08-16 18:51:45,ENVIO,9083442797,20911.689453,ACH,0
18166019,2023-08-16 21:11:21,ENVIO,6673377477,204.690002,ACH,0


In [12]:
import numpy as np
import pandas as pd

print("Preparando características dos eventos...")

# 1. Valor da transação em escala logarítmica
eventos_conta["log_amount"] = np.log1p(
    eventos_conta["amount"]
).astype("float32")

# 2. Intervalo desde o evento anterior da mesma conta
intervalo = (
    eventos_conta.groupby("account_id", sort=False)["timestamp"]
    .diff()
)

segundos = intervalo.dt.total_seconds()

# Indicador de existência de evento anterior
eventos_conta["has_previous"] = (
    intervalo.notna()
).astype("int8")

# Transformação logarítmica do intervalo
eventos_conta["gap_log"] = np.log1p(
    segundos.fillna(0).clip(lower=0)
).astype("float32")

del intervalo, segundos

# 3. Características cíclicas do horário
hora = eventos_conta["timestamp"].dt.hour.to_numpy(
    dtype=np.float32
)

eventos_conta["hour_sin"] = np.sin(
    2 * np.pi * hora / 24
).astype("float32")

eventos_conta["hour_cos"] = np.cos(
    2 * np.pi * hora / 24
).astype("float32")

# 4. Características cíclicas do dia da semana
dia = eventos_conta["timestamp"].dt.day_of_week.to_numpy(
    dtype=np.float32
)

eventos_conta["weekday_sin"] = np.sin(
    2 * np.pi * dia / 7
).astype("float32")

eventos_conta["weekday_cos"] = np.cos(
    2 * np.pi * dia / 7
).astype("float32")

print("Características criadas!")

# Conferir os resultados
colunas_verificacao = [
    "account_id",
    "timestamp",
    "amount",
    "log_amount",
    "direction",
    "gap_log",
    "has_previous",
    "hour_sin",
    "hour_cos",
    "weekday_sin",
    "weekday_cos",
    "payment_type",
    "y"
]

display(
    eventos_conta[colunas_verificacao]
    .sample(10, random_state=42)
)

print("\nValores ausentes nas novas características:")

display(
    eventos_conta[
        [
            "log_amount",
            "gap_log",
            "has_previous",
            "hour_sin",
            "hour_cos",
            "weekday_sin",
            "weekday_cos"
        ]
    ].isna().sum()
)

Preparando características dos eventos...
Características criadas!


,account_id,timestamp,amount,log_amount,direction,gap_log,has_previous,hour_sin,hour_cos,weekday_sin,weekday_cos,payment_type,y
4761921,2519514724,2023-06-12 16:58:13,6909.240234,8.840759,1,4.543295,1,-0.866025,-4.999999e-01,0.000000,1.000000,4,0
12056254,6343106098,2022-11-16 18:46:47,20571.080078,9.931690,1,8.791486,1,-1.000000,1.192488e-08,0.974928,-0.222521,4,0
1897111,1008114820,2023-03-26 17:16:48,4575.740234,8.428742,0,2.397895,1,-0.965926,-2.588190e-01,-0.781832,0.623490,6,0
14458985,7610760141,2022-12-18 13:14:41,6470.569824,8.775174,0,5.579730,1,-0.258819,-9.659258e-01,-0.781832,0.623490,6,0
4964470,2627841996,2023-04-15 20:02:40,23.990000,3.218476,1,11.570100,1,-0.866025,5.000003e-01,-0.974928,-0.222521,6,0
5210535,2759009808,2023-01-02 00:09:34,161.649994,5.091600,0,14.217612,1,0.000000,1.000000e+00,0.000000,1.000000,0,0
11116517,5847649137,2023-02-11 11:46:55,24131.429688,10.091311,0,9.190546,1,0.258819,-9.659259e-01,-0.974928,-0.222521,5,0
11566597,6083953045,2023-06-27 19:07:14,118.000000,4.779123,0,14.832313,1,-0.965926,2.588190e-01,0.781832,0.623490,0,0
15341611,8076812096,2023-05-31 10:25:52,9113.780273,9.117653,1,4.983607,1,0.500000,-8.660255e-01,0.974928,-0.222521,6,0
11435538,6016900593,2023-06-26 20:12:18,6104.399902,8.716928,0,5.411646,1,-0.866025,5.000003e-01,0.000000,1.000000,0,0



Valores ausentes nas novas características:


log_amount      0
gap_log         0
has_previous    0
hour_sin        0
hour_cos        0
weekday_sin     0
weekday_cos     0
dtype: int64

In [13]:
categorias_pagamento = df["Payment_type"].cat.categories

print("=== CODIFICAÇÃO DOS TIPOS DE PAGAMENTO ===")

for codigo, nome in enumerate(categorias_pagamento):
    print(f"{codigo}: {nome}")

=== CODIFICAÇÃO DOS TIPOS DE PAGAMENTO ===
0: ACH
1: Cash Deposit
2: Cash Withdrawal
3: Cheque
4: Credit card
5: Cross-border
6: Debit card


In [14]:
from sklearn.preprocessing import StandardScaler

# Características numéricas que entrarão na rede
features_numericas = [
    "log_amount",
    "direction",
    "gap_log",
    "has_previous",
    "hour_sin",
    "hour_cos",
    "weekday_sin",
    "weekday_cos"
]

# Categorias de pagamento
features_categoricas = ["payment_type"]

# Separação temporal dos eventos
inicio_treino = pd.Timestamp("2023-05-19")

mask_treino = (
    eventos_conta["timestamp"] < inicio_treino
).to_numpy()

# Converter características numéricas para matriz float32
X_numerico = eventos_conta[
    features_numericas
].to_numpy(dtype=np.float32)

# Extrair os códigos categóricos
X_pagamento = eventos_conta[
    "payment_type"
].to_numpy(dtype=np.int64)

# Conferir se todos os códigos são válidos
assert X_pagamento.min() >= 0
assert X_pagamento.max() < len(categorias_pagamento)

# Ajustar o scaler somente no treino
scaler = StandardScaler()

scaler.fit(X_numerico[mask_treino])

# Aplicar a transformação em todos os períodos
X_numerico = scaler.transform(
    X_numerico
).astype(np.float32)

# Manter os rótulos separados das entradas
y_eventos = eventos_conta["y"].to_numpy(dtype=np.int64)

print("=== PREPARAÇÃO CONCLUÍDA ===")
print("Matriz numérica:", X_numerico.shape)
print("Matriz categórica:", X_pagamento.shape)
print("Rótulos:", y_eventos.shape)

print("\nQuantidade de características numéricas:",
      len(features_numericas))

print("Quantidade de categorias:",
      len(categorias_pagamento))

print("\nValores ausentes na matriz numérica:",
      np.isnan(X_numerico).sum())

=== PREPARAÇÃO CONCLUÍDA ===
Matriz numérica: (19009704, 8)
Matriz categórica: (19009704,)
Rótulos: (19009704,)

Quantidade de características numéricas: 8
Quantidade de categorias: 7

Valores ausentes na matriz numérica: 0


In [15]:

# ============================================================
# CONSTRUÇÃO DOS TENSORES DE TREINO, VALIDAÇÃO E TESTE
# ============================================================

import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

# Garantir tamanho de janela
TAMANHO_JANELA = 10

# Separação temporal dos eventos
mascaras = {
    "treino": (
        eventos_conta["timestamp"] < pd.Timestamp("2023-05-19")
    ).to_numpy(),

    "validacao": (
        (eventos_conta["timestamp"] >= pd.Timestamp("2023-05-19")) &
        (eventos_conta["timestamp"] < pd.Timestamp("2023-07-06"))
    ).to_numpy(),

    "teste": (
        eventos_conta["timestamp"] >= pd.Timestamp("2023-07-06")
    ).to_numpy()
}

# Quantidade de janelas que já havíamos criado
quantidades_esperadas = {
    "treino": 1_078_404,
    "validacao": 194_016,
    "teste": 190_825
}

# Reconstruir os índices dos eventos de cada janela,
# preservando a ordem cronológica e descartando apenas
# as sobras incompletas já descartadas anteriormente.

indices_por_periodo = {}

for nome, mascara in mascaras.items():

    indices_eventos = np.flatnonzero(mascara)

    parte = eventos_conta.loc[
        mascara, ["account_id"]
    ].copy()

    parte["_idx_evento"] = indices_eventos

    parte["_posicao"] = (
        parte.groupby("account_id", sort=False).cumcount()
    )

    total_conta = (
        parte.groupby("account_id", sort=False)["_idx_evento"]
        .transform("size")
    )

    # Manter somente grupos completos de 10 eventos
    parte = parte.loc[
        parte["_posicao"] <
        (total_conta // TAMANHO_JANELA) * TAMANHO_JANELA
    ]

    indices = parte["_idx_evento"].to_numpy(
        dtype=np.int32
    ).reshape(-1, TAMANHO_JANELA)

    assert len(indices) == quantidades_esperadas[nome], (
        f"Quantidade inesperada de janelas em {nome}"
    )

    indices_por_periodo[nome] = indices

# Construir tensores
tensores = {}

for nome, indices in indices_por_periodo.items():

    # Matriz numérica: (janelas, 10 eventos, 8 características)
    X_num = np.ascontiguousarray(
        X_numerico[indices], dtype=np.float32
    )

    # Matriz categórica: (janelas, 10 eventos)
    X_cat = np.ascontiguousarray(
        X_pagamento[indices], dtype=np.int64
    )

    # Rótulo da janela: 1 se algum evento for suspeito
    y = np.ascontiguousarray(
        y_eventos[indices].max(axis=1),
        dtype=np.float32
    )

    tensores[nome] = (
        torch.from_numpy(X_num),
        torch.from_numpy(X_cat),
        torch.from_numpy(y)
    )

# Criar datasets
datasets = {
    nome: TensorDataset(*dados)
    for nome, dados in tensores.items()
}

# Criar DataLoaders
TAMANHO_BATCH = 1024

train_loader = DataLoader(
    datasets["treino"],
    batch_size=TAMANHO_BATCH,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    datasets["validacao"],
    batch_size=TAMANHO_BATCH,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    datasets["teste"],
    batch_size=TAMANHO_BATCH,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

# Verificação final
print("=== TENSORES CONSTRUÍDOS ===")

for nome, (X_num, X_cat, y) in tensores.items():
    print(f"\n{nome.upper()}")
    print("Numérico:", tuple(X_num.shape))
    print("Categórico:", tuple(X_cat.shape))
    print("Rótulos:", tuple(y.shape))
    print("Janelas suspeitas:", int(y.sum().item()))
    print("Janelas normais:", int((y == 0).sum().item()))

print("\n=== DATA LOADERS ===")
print("Batches de treino:", len(train_loader))
print("Batches de validação:", len(val_loader))
print("Batches de teste:", len(test_loader))

print("\nTotal de janelas:",
      sum(len(d) for d in datasets.values()))

print("\nGPU CUDA disponível:", torch.cuda.is_available())


=== TENSORES CONSTRUÍDOS ===

TREINO
Numérico: (1078404, 10, 8)
Categórico: (1078404, 10)
Rótulos: (1078404,)
Janelas suspeitas: 5580
Janelas normais: 1072824

VALIDACAO
Numérico: (194016, 10, 8)
Categórico: (194016, 10)
Rótulos: (194016,)
Janelas suspeitas: 683
Janelas normais: 193333

TESTE
Numérico: (190825, 10, 8)
Categórico: (190825, 10)
Rótulos: (190825,)
Janelas suspeitas: 786
Janelas normais: 190039

=== DATA LOADERS ===
Batches de treino: 1054
Batches de validação: 190
Batches de teste: 187

Total de janelas: 1463245

GPU CUDA disponível: False


In [16]:
import copy
import numpy as np
import torch
import torch.nn as nn

from sklearn.metrics import average_precision_score

# ============================================================
# 1. CONFIGURAÇÕES
# ============================================================

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

dispositivo = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Dispositivo:", dispositivo)

# Peso da classe positiva, calculado somente no treino
X_num_treino, X_cat_treino, y_treino = tensores["treino"]

quantidade_positivos = int(y_treino.sum().item())
quantidade_negativos = len(y_treino) - quantidade_positivos

peso_positivo = quantidade_negativos / quantidade_positivos

print("Janelas positivas:", quantidade_positivos)
print("Janelas negativas:", quantidade_negativos)
print("Pos weight:", round(peso_positivo, 4))


# ============================================================
# 2. ARQUITETURA LSTM
# ============================================================

class FraudRadarLSTM(nn.Module):

    def __init__(
        self,
        caracteristicas_numericas=8,
        categorias_pagamento=7,
        dimensao_embedding=4,
        hidden_size=128,
        num_layers=2,
        dropout=0.20
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=categorias_pagamento,
            embedding_dim=dimensao_embedding
        )

        self.lstm = nn.LSTM(
            input_size=caracteristicas_numericas + dimensao_embedding,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        self.classificador = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x_numerico, x_categorico):

        # Representação categórica
        x_embedding = self.embedding(x_categorico)

        # Combinar características numéricas e categóricas
        x = torch.cat(
            [x_numerico, x_embedding],
            dim=2
        )

        # Processar sequência temporal
        saida, (estado_oculto, estado_celula) = self.lstm(x)

        # Representação final da sequência
        representacao = estado_oculto[-1]

        # Classificação da janela
        logits = self.classificador(representacao)

        return logits.squeeze(1)


# ============================================================
# 3. INICIALIZAR MODELO
# ============================================================

modelo_lstm = FraudRadarLSTM().to(dispositivo)

quantidade_parametros = sum(
    p.numel() for p in modelo_lstm.parameters()
    if p.requires_grad
)

print("\n=== MODELO LSTM ===")
print(modelo_lstm)
print("\nParâmetros treináveis:", f"{quantidade_parametros:,}")


# ============================================================
# 4. FUNÇÃO DE PERDA E OTIMIZADOR
# ============================================================

criterio = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        [peso_positivo],
        dtype=torch.float32,
        device=dispositivo
    )
)

otimizador = torch.optim.AdamW(
    modelo_lstm.parameters(),
    lr=0.001,
    weight_decay=0.00001
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    otimizador,
    mode="max",
    factor=0.5,
    patience=2,
    min_lr=0.00001
)


# ============================================================
# 5. FUNÇÃO DE AVALIAÇÃO
# ============================================================

def avaliar_modelo(modelo, loader):

    modelo.eval()

    perdas = 0.0
    total = 0

    todas_previsoes = []
    todos_rotulos = []

    with torch.no_grad():

        for x_num, x_cat, y in loader:

            x_num = x_num.to(dispositivo)
            x_cat = x_cat.to(dispositivo)
            y = y.to(dispositivo)

            logits = modelo(x_num, x_cat)

            perda = criterio(logits, y)

            perdas += perda.item() * len(y)
            total += len(y)

            todas_previsoes.append(
                logits.cpu().numpy()
            )

            todos_rotulos.append(
                y.cpu().numpy()
            )

    previsoes = np.concatenate(todas_previsoes)
    rotulos = np.concatenate(todos_rotulos)

    ap = average_precision_score(
        rotulos,
        previsoes
    )

    perda_media = perdas / total

    return perda_media, ap


# ============================================================
# 6. TREINAMENTO
# ============================================================

EPOCAS = 20
PACIENCIA = 4

melhor_ap = -1.0
melhores_pesos = None
epocas_sem_melhoria = 0

historico_lstm = []

print("\n=== INICIANDO TREINAMENTO LSTM ===")

for epoca in range(1, EPOCAS + 1):

    modelo_lstm.train()

    perda_total = 0.0
    total_amostras = 0

    for x_num, x_cat, y in train_loader:

        x_num = x_num.to(dispositivo)
        x_cat = x_cat.to(dispositivo)
        y = y.to(dispositivo)

        otimizador.zero_grad(set_to_none=True)

        logits = modelo_lstm(x_num, x_cat)

        perda = criterio(logits, y)

        perda.backward()

        # Estabilizar o treinamento recorrente
        nn.utils.clip_grad_norm_(
            modelo_lstm.parameters(),
            max_norm=1.0
        )

        otimizador.step()

        perda_total += perda.item() * len(y)
        total_amostras += len(y)

    perda_treino = perda_total / total_amostras

    # Avaliação na validação
    perda_validacao, ap_validacao = avaliar_modelo(
        modelo_lstm,
        val_loader
    )

    scheduler.step(ap_validacao)

    taxa_aprendizado = otimizador.param_groups[0]["lr"]

    historico_lstm.append({
        "epoca": epoca,
        "perda_treino": perda_treino,
        "perda_validacao": perda_validacao,
        "ap_validacao": ap_validacao,
        "learning_rate": taxa_aprendizado
    })

    print(
        f"Época {epoca:02d}/{EPOCAS} | "
        f"Loss treino: {perda_treino:.5f} | "
        f"Loss validação: {perda_validacao:.5f} | "
        f"AP validação: {ap_validacao:.5f} | "
        f"LR: {taxa_aprendizado:.6f}"
    )

    # Salvar melhor modelo pela AP de validação
    if ap_validacao > melhor_ap + 0.00001:

        melhor_ap = ap_validacao

        melhores_pesos = copy.deepcopy(
            modelo_lstm.state_dict()
        )

        epocas_sem_melhoria = 0

    else:

        epocas_sem_melhoria += 1

    if epocas_sem_melhoria >= PACIENCIA:

        print("\nParada antecipada: validação sem melhoria.")
        break


# ============================================================
# 7. RESTAURAR MELHOR MODELO
# ============================================================

modelo_lstm.load_state_dict(melhores_pesos)
modelo_lstm.eval()

print("\n=== TREINAMENTO LSTM CONCLUÍDO ===")
print("Melhor AP na validação:", round(melhor_ap, 5))
print("Épocas executadas:", len(historico_lstm))


Dispositivo: cpu
Janelas positivas: 5580
Janelas negativas: 1072824
Pos weight: 192.2624

=== MODELO LSTM ===
FraudRadarLSTM(
  (embedding): Embedding(7, 4)
  (lstm): LSTM(12, 128, num_layers=2, batch_first=True, dropout=0.2)
  (classificador): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
  )
)

Parâmetros treináveis: 213,149

=== INICIANDO TREINAMENTO LSTM ===
Época 01/20 | Loss treino: 0.81083 | Loss validação: 0.67021 | AP validação: 0.04925 | LR: 0.001000
Época 02/20 | Loss treino: 0.67789 | Loss validação: 0.68287 | AP validação: 0.06335 | LR: 0.001000
Época 03/20 | Loss treino: 0.62550 | Loss validação: 0.66665 | AP validação: 0.09844 | LR: 0.001000
Época 04/20 | Loss treino: 0.59418 | Loss validação: 0.58777 | AP validação: 0.15359 | LR: 0.001000
Época 05/20 | Loss treino: 0.56237 | Loss validação: 0.56351 | AP validação: 0.18010 | LR: 0.0

In [17]:
import copy
import numpy as np
import torch
import torch.nn as nn,

from sklearn.metrics import average_precision_score

# ============================================================
# 1. ARQUITETURA GRU
# ============================================================

class FraudRadarGRU(nn.Module):

    def __init__(
        self,
        caracteristicas_numericas=8,
        categorias_pagamento=7,
        dimensao_embedding=4,
        hidden_size=128,
        num_layers=2,
        dropout=0.20
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=categorias_pagamento,
            embedding_dim=dimensao_embedding
        )

        self.gru = nn.GRU(
            input_size=caracteristicas_numericas + dimensao_embedding,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        self.classificador = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x_numerico, x_categorico):

        x_embedding = self.embedding(x_categorico)

        x = torch.cat(
            [x_numerico, x_embedding],
            dim=2
        )

        saida, estado_oculto = self.gru(x)

        representacao = estado_oculto[-1]

        logits = self.classificador(representacao)

        return logits.squeeze(1)


# ============================================================
# 2. INICIALIZAÇÃO
# ============================================================

SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

modelo_gru = FraudRadarGRU().to(dispositivo)

quantidade_parametros_gru = sum(
    p.numel() for p in modelo_gru.parameters()
    if p.requires_grad
)

print("=== MODELO GRU ===")
print(modelo_gru)

print(
    "\nParâmetros treináveis:",
    f"{quantidade_parametros_gru:,}"
)


# ============================================================
# 3. FUNÇÃO DE AVALIAÇÃO
# ============================================================

criterio_gru = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        [peso_positivo],
        dtype=torch.float32,
        device=dispositivo
    )
)


def avaliar_gru(modelo, loader):

    modelo.eval()

    perda_total = 0.0
    total = 0

    todas_previsoes = []
    todos_rotulos = []

    with torch.no_grad():

        for x_num, x_cat, y in loader:

            x_num = x_num.to(dispositivo)
            x_cat = x_cat.to(dispositivo)
            y = y.to(dispositivo)

            logits = modelo(x_num, x_cat)

            perda = criterio_gru(logits, y)

            perda_total += perda.item() * len(y)
            total += len(y)

            todas_previsoes.append(
                logits.cpu().numpy()
            )

            todos_rotulos.append(
                y.cpu().numpy()
            )

    previsoes = np.concatenate(todas_previsoes)
    rotulos = np.concatenate(todos_rotulos)

    ap = average_precision_score(
        rotulos,
        previsoes
    )

    return perda_total / total, ap


# ============================================================
# 4. OTIMIZADOR
# ============================================================

otimizador_gru = torch.optim.AdamW(
    modelo_gru.parameters(),
    lr=0.001,
    weight_decay=0.00001
)

scheduler_gru = torch.optim.lr_scheduler.ReduceLROnPlateau(
    otimizador_gru,
    mode="max",
    factor=0.5,
    patience=2,
    min_lr=0.00001
)


# ============================================================
# 5. TREINAMENTO
# ============================================================

EPOCAS = 20
PACIENCIA = 4

melhor_ap_gru = -1.0
melhores_pesos_gru = None
epocas_sem_melhoria_gru = 0

historico_gru = []

print("\n=== INICIANDO TREINAMENTO GRU ===")

for epoca in range(1, EPOCAS + 1):

    modelo_gru.train()

    perda_total = 0.0
    total_amostras = 0

    for x_num, x_cat, y in train_loader:

        x_num = x_num.to(dispositivo)
        x_cat = x_cat.to(dispositivo)
        y = y.to(dispositivo)

        otimizador_gru.zero_grad(set_to_none=True)

        logits = modelo_gru(x_num, x_cat)

        perda = criterio_gru(logits, y)

        perda.backward()

        nn.utils.clip_grad_norm_(
            modelo_gru.parameters(),
            max_norm=1.0
        )

        otimizador_gru.step()

        perda_total += perda.item() * len(y)
        total_amostras += len(y)

    perda_treino = perda_total / total_amostras

    perda_validacao, ap_validacao = avaliar_gru(
        modelo_gru,
        val_loader
    )

    scheduler_gru.step(ap_validacao)

    taxa_aprendizado = otimizador_gru.param_groups[0]["lr"]

    historico_gru.append({
        "epoca": epoca,
        "perda_treino": perda_treino,
        "perda_validacao": perda_validacao,
        "ap_validacao": ap_validacao,
        "learning_rate": taxa_aprendizado
    })

    print(
        f"Época {epoca:02d}/{EPOCAS} | "
        f"Loss treino: {perda_treino:.5f} | "
        f"Loss validação: {perda_validacao:.5f} | "
        f"AP validação: {ap_validacao:.5f} | "
        f"LR: {taxa_aprendizado:.6f}"
    )

    if ap_validacao > melhor_ap_gru + 0.00001:

        melhor_ap_gru = ap_validacao

        melhores_pesos_gru = copy.deepcopy(
            modelo_gru.state_dict()
        )

        epocas_sem_melhoria_gru = 0

    else:

        epocas_sem_melhoria_gru += 1

    if epocas_sem_melhoria_gru >= PACIENCIA:

        print("\nParada antecipada: validação sem melhoria.")
        break


# ============================================================
# 6. RESTAURAR MELHOR MODELO
# ============================================================

modelo_gru.load_state_dict(melhores_pesos_gru)
modelo_gru.eval()

print("\n=== TREINAMENTO GRU CONCLUÍDO ===")
print("Melhor AP na validação:", round(melhor_ap_gru, 5))
print("Épocas executadas:", len(historico_gru))


=== MODELO GRU ===
FraudRadarGRU(
  (embedding): Embedding(7, 4)
  (gru): GRU(12, 128, num_layers=2, batch_first=True, dropout=0.2)
  (classificador): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
  )
)

Parâmetros treináveis: 161,949

=== INICIANDO TREINAMENTO GRU ===
Época 01/20 | Loss treino: 0.78855 | Loss validação: 0.65570 | AP validação: 0.05978 | LR: 0.001000
Época 02/20 | Loss treino: 0.66249 | Loss validação: 0.61538 | AP validação: 0.08614 | LR: 0.001000
Época 03/20 | Loss treino: 0.60279 | Loss validação: 0.56383 | AP validação: 0.11028 | LR: 0.001000
Época 04/20 | Loss treino: 0.55482 | Loss validação: 0.53871 | AP validação: 0.11537 | LR: 0.001000
Época 05/20 | Loss treino: 0.53207 | Loss validação: 0.55656 | AP validação: 0.17880 | LR: 0.001000
Época 06/20 | Loss treino: 0.50388 | Loss validação: 0.53021 | AP validação: 0.18920 | LR

In [18]:
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import average_precision_score

# ============================================================
# 1. OBTER AS PONTUAÇÕES DOS MODELOS
# ============================================================

def obter_previsoes(modelo, loader):

    modelo.eval()

    todas_pontuacoes = []
    todos_rotulos = []

    with torch.no_grad():

        for x_num, x_cat, y in loader:

            x_num = x_num.to(dispositivo)
            x_cat = x_cat.to(dispositivo)

            logits = modelo(x_num, x_cat)

            probabilidades = torch.sigmoid(logits)

            todas_pontuacoes.append(
                probabilidades.cpu().numpy()
            )

            todos_rotulos.append(
                y.numpy()
            )

    pontuacoes = np.concatenate(todas_pontuacoes)
    rotulos = np.concatenate(todos_rotulos)

    return pontuacoes, rotulos


# ============================================================
# 2. PREVISÕES NA VALIDAÇÃO E NO TESTE
# ============================================================

print("Obtendo previsões da LSTM...")

scores_lstm_val, y_val = obter_previsoes(
    modelo_lstm,
    val_loader
)

scores_lstm_test, y_test = obter_previsoes(
    modelo_lstm,
    test_loader
)

print("Obtendo previsões da GRU...")

scores_gru_val, y_val_gru = obter_previsoes(
    modelo_gru,
    val_loader
)

scores_gru_test, y_test_gru = obter_previsoes(
    modelo_gru,
    test_loader
)

# Garantir que a ordem dos rótulos seja idêntica
assert np.array_equal(y_val, y_val_gru)
assert np.array_equal(y_test, y_test_gru)


# ============================================================
# 3. AVALIAÇÃO GERAL
# ============================================================

resultados_ap = []

for nome, scores_val, scores_test in [
    ("LSTM", scores_lstm_val, scores_lstm_test),
    ("GRU", scores_gru_val, scores_gru_test)
]:

    ap_val = average_precision_score(
        y_val,
        scores_val
    )

    ap_test = average_precision_score(
        y_test,
        scores_test
    )

    resultados_ap.append({
        "Modelo": nome,
        "AP Validação": ap_val,
        "AP Teste": ap_test
    })


# ============================================================
# 4. PRECISÃO E RECALL NO TOP-K
# ============================================================

def avaliar_top_k(rotulos, pontuacoes, valores_k):

    indices_ordenados = np.argsort(
        pontuacoes
    )[::-1]

    total_suspeitos = int(rotulos.sum())

    resultados = []

    for k in valores_k:

        indices_top = indices_ordenados[:k]

        verdadeiros_positivos = int(
            rotulos[indices_top].sum()
        )

        precisao = verdadeiros_positivos / k

        recall = (
            verdadeiros_positivos / total_suspeitos
            if total_suspeitos > 0 else 0
        )

        resultados.append({
            "Top K": k,
            "Suspeitas encontradas": verdadeiros_positivos,
            "Precisão": precisao,
            "Recall": recall
        })

    return resultados


valores_k = [100, 300, 1000, 3000]

resultados_top_k = []

for nome, scores_val, scores_test in [
    ("LSTM", scores_lstm_val, scores_lstm_test),
    ("GRU", scores_gru_val, scores_gru_test)
]:

    for conjunto, rotulos, scores in [
        ("Validação", y_val, scores_val),
        ("Teste", y_test, scores_test)
    ]:

        resultados = avaliar_top_k(
            rotulos,
            scores,
            valores_k
        )

        for resultado in resultados:

            resultados_top_k.append({
                "Modelo": nome,
                "Conjunto": conjunto,
                **resultado
            })


# ============================================================
# 5. EXIBIR RESULTADOS
# ============================================================

df_ap = pd.DataFrame(resultados_ap)

df_top_k = pd.DataFrame(resultados_top_k)

print("\n==============================")
print("AVERAGE PRECISION")
print("==============================")

print(
    df_ap.to_string(
        index=False,
        formatters={
            "AP Validação": "{:.5f}".format,
            "AP Teste": "{:.5f}".format
        }
    )
)

print("\n==============================")
print("RESULTADOS TOP-K")
print("==============================")

print(
    df_top_k.to_string(
        index=False,
        formatters={
            "Precisão": "{:.2%}".format,
            "Recall": "{:.2%}".format
        }
    )
)

print("\n==============================")
print("DISTRIBUIÇÃO DO TESTE")
print("==============================")

print("Total de janelas:", len(y_test))
print("Janelas suspeitas:", int(y_test.sum()))
print("Janelas normais:", int((y_test == 0).sum()))
print("Proporção suspeita:", f"{y_test.mean():.4%}")


Obtendo previsões da LSTM...
Obtendo previsões da GRU...

AVERAGE PRECISION
Modelo AP Validação AP Teste
  LSTM      0.22582  0.23465
   GRU      0.21113  0.22788

RESULTADOS TOP-K
Modelo  Conjunto  Top K  Suspeitas encontradas Precisão Recall
  LSTM Validação    100                     84   84.00% 12.30%
  LSTM Validação    300                    130   43.33% 19.03%
  LSTM Validação   1000                    185   18.50% 27.09%
  LSTM Validação   3000                    269    8.97% 39.39%
  LSTM     Teste    100                     99   99.00% 12.60%
  LSTM     Teste    300                    152   50.67% 19.34%
  LSTM     Teste   1000                    217   21.70% 27.61%
  LSTM     Teste   3000                    294    9.80% 37.40%
   GRU Validação    100                     77   77.00% 11.27%
   GRU Validação    300                    135   45.00% 19.77%
   GRU Validação   1000                    183   18.30% 26.79%
   GRU Validação   3000                    253    8.43% 37.04%


In [19]:
print("=== COMPARAÇÃO COMPLETA NO TESTE ===")

print(
    df_top_k[df_top_k["Conjunto"] == "Teste"].to_string(
        index=False,
        formatters={
            "Precisão": "{:.2%}".format,
            "Recall": "{:.2%}".format
        }
    )
)

print("\n=== AVERAGE PRECISION ===")

print(
    df_ap.to_string(
        index=False,
        formatters={
            "AP Validação": "{:.5f}".format,
            "AP Teste": "{:.5f}".format
        }
    )
)


=== COMPARAÇÃO COMPLETA NO TESTE ===
Modelo Conjunto  Top K  Suspeitas encontradas Precisão Recall
  LSTM    Teste    100                     99   99.00% 12.60%
  LSTM    Teste    300                    152   50.67% 19.34%
  LSTM    Teste   1000                    217   21.70% 27.61%
  LSTM    Teste   3000                    294    9.80% 37.40%
   GRU    Teste    100                     89   89.00% 11.32%
   GRU    Teste    300                    145   48.33% 18.45%
   GRU    Teste   1000                    213   21.30% 27.10%
   GRU    Teste   3000                    294    9.80% 37.40%

=== AVERAGE PRECISION ===
Modelo AP Validação AP Teste
  LSTM      0.22582  0.23465
   GRU      0.21113  0.22788
